In [4]:
from __future__ import annotations

import math
from dataclasses import dataclass
from pathlib import Path
from typing import List, Tuple
import os
from openai import AzureOpenAI
import numpy as np

# 2.1 Configure Paramters of RAG 
(Retrevial Augmented Generation)

In [ ]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
if not OPENAI_API_KEY:
    raise RuntimeError("OPENAI_API_KEY is not set.")
os.environ['AZURE_OPENAI_API_KEY'] = OPENAI_API_KEY
os.environ['AZURE_OPENAI_ENDPOINT'] = 'http://pluralsight.openai.azure.com'
os.environ['AZURE_OPENAI_API_VERSION'] = '2024-06-01'
DOCUMENT_PATH = "rag_example_knowledge_base.txt"

# Embedding model for semantic search
EMBEDDING_MODEL = "text-embedding-3-small"

# Generation model for final answer
RESPONSE_MODEL = "gpt-5-mini"

# Chunking settings
CHUNK_SIZE = 1000          # characters
CHUNK_OVERLAP = 150        # characters
TOP_K = 4                  # number of chunks to retrieve

embedding_client = AzureOpenAI(
    api_key=OPENAI_API_KEY,
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    base_url="http://pluralsight.openai.azure.com/openai/v1/",
)
response_client = AzureOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),  
)

# 3 Read File

In [15]:
def read_text_file(path: str) -> str:
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"File not found: {path}")

    return p.read_text(encoding="utf-8")

# 3.1 Index creation

In [16]:
@dataclass
class ChunkRecord:
    chunk_id: int
    text: str
    embedding: np.ndarray

def build_in_memory_index(document_text: str) -> List[ChunkRecord]:
    """
    Chunk the document and embed each chunk.
    """
    chunks = chunk_text(
        text=document_text,
        chunk_size=CHUNK_SIZE,
        overlap=CHUNK_OVERLAP
    )

    print(f"Created {len(chunks)} chunks.")

    embeddings = embed_texts(chunks, EMBEDDING_MODEL)

    index = []
    for i, (chunk, embedding) in enumerate(zip(chunks, embeddings)):
        index.append(
            ChunkRecord(
                chunk_id=i,
                text=chunk,
                embedding=embedding
            )
        )

    return index

# 3.2 Chunking

In [17]:
def chunk_text(text: str, chunk_size: int, overlap: int) -> List[str]:
    """
    Splits text into overlapping character-based chunks.

    For production, token-based chunking is often better, but this is a
    simple and readable demo.
    """
    if chunk_size <= 0:
        raise ValueError("chunk_size must be > 0")
    if overlap < 0:
        raise ValueError("overlap must be >= 0")
    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")

    chunks = []
    start = 0
    text_len = len(text)

    while start < text_len:
        end = min(start + chunk_size, text_len)
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)

        if end == text_len:
            break

        start += chunk_size - overlap

    return chunks



# 3.3 Embeddings

In [18]:
def embed_texts(texts: List[str], model: str) -> List[np.ndarray]:
    """
    Batch-embed a list of strings using OpenAI's embeddings API.
    """
    if not texts:
        return []

    response = embedding_client.embeddings.create(
        input = texts,
        model= "text-embedding-3-small"
    )

    # The API returns one embedding per input item.
    vectors = [np.array(item.embedding, dtype=np.float32) for item in response.data]
    return vectors


def embed_single_text(text: str, model: str) -> np.ndarray:
    response = embedding_client.embeddings.create(
        model=model,
        input=text
    )
    return np.array(response.data[0].embedding, dtype=np.float32)

# 4.1 Similarity search

In [19]:
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """
    Cosine similarity between two vectors.
    """
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom)


def retrieve_top_k(
    query_embedding: np.ndarray,
    index: List[ChunkRecord],
    top_k: int
) -> List[Tuple[ChunkRecord, float]]:
    """
    Score every chunk against the query and return the top_k by similarity.
    """
    scored = []
    for record in index:
        score = cosine_similarity(query_embedding, record.embedding)
        scored.append((record, score))

    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_k]



# 5.1 Prompt building

In [20]:
def build_context(retrieved: List[Tuple[ChunkRecord, float]]) -> str:
    """
    Formats retrieved chunks into a context block.
    """
    sections = []
    for record, score in retrieved:
        sections.append(
            f"[Chunk {record.chunk_id} | similarity={score:.4f}]\n{record.text}"
        )
    return "\n\n" + ("\n\n" + ("-" * 80) + "\n\n").join(sections)


def answer_question_with_rag(question: str, retrieved: list):
    """
    Generate an answer using Azure OpenAI chat completions
    and retrieved document chunks.
    """

    # Build context string from retrieved chunks
    context_sections = []
    for record, score in retrieved:
        context_sections.append(
            f"[Chunk {record.chunk_id} | similarity={score:.4f}]\n{record.text}"
        )

    context = "\n\n".join(context_sections)

    prompt = f"""
You are answering questions using retrieved context from a document.

Rules:
- Use only the provided context
- If the answer is not present, say "I couldn't find that in the document."
- Be concise but complete.

Context:
{context}


Question:
{question}
""".strip()

    response = response_client.chat.completions.create( 
        model="gpt-4o-mini", 
        messages=[
            {
                "role": "system",
                "content": "You answer questions using provided document context."
            },
            {
                "role": "user",
                "content": prompt
            }
        ]
    ) 
    return response.choices[0].message.content

# 5.2 RAG Driver Demo

In [21]:
"""
Self-managed RAG example with NO OpenAI Files API.

What this does:
1. Reads a local text document
2. Splits it into chunks
3. Embeds each chunk with OpenAI embeddings
4. Stores chunk text + embeddings in memory
5. Embeds a user question
6. Finds the most similar chunks with cosine similarity
7. Sends the retrieved chunks to the Responses API
8. Gets an answer grounded in the retrieved context
"""

def run_demo():
    document_text = read_text_file(DOCUMENT_PATH)
    print(f"Loaded document with {len(document_text)} characters.")

    # Build searchable vector index
    index = build_in_memory_index(document_text)

    # Example questions
    questions = [
        "What does the support policy say?",
        "Which product helps with predictive models?",
        "Can dashboards be exported?",
        "What encryption is used at rest?",
        "Does the document mention GPU training?"
    ]

    for question in questions:
        print("\n" + "=" * 100)
        print(f"QUESTION: {question}")

        # Embed the question
        query_embedding = embed_single_text(question, EMBEDDING_MODEL)

        # Retrieve most relevant chunks
        retrieved = retrieve_top_k(
            query_embedding=query_embedding,
            index=index,
            top_k=TOP_K
        )

        print("\nTop retrieved chunks:")
        for record, score in retrieved:
            preview = record.text[:140].replace("\n", " ")
            print(f"  - Chunk {record.chunk_id} | score={score:.4f} | {preview}...")

        # Generate final answer using only retrieved context
        answer = answer_question_with_rag(question, retrieved)

        print("\nANSWER:")
        print(answer)


In [22]:
run_demo()

Loaded document with 1490 characters.
Created 2 chunks.


APIConnectionError: Connection error.